# Fase 12 — Validación de recomendaciones estratégicas para Golazo

Comprueba, con datos y con criterio de negocio propio de un canal de fútbol, la batería de recomendaciones planteadas para el cliente. Pensado para alimentar directamente una presentación / demo en Streamlit: cada sección deja al menos una tabla o gráfico listo para reutilizar.

**Matriz de viabilidad** (resumen; detalle en la conversación):

| # | Hipótesis | Viabilidad |
|---|---|---|
| 1 | Shorts-Opinión justo tras el partido | Parcial (proxy por día de semana, sin hora de publicación) |
| 2 | Largo-Seguimiento el día después | Sí |
| 3 | Curiosidades de fútbol a fondo + comparación con Afición | Sí |
| 4 | Shorts-Afición como relleno de bajo coste | Parcial (solo se confirma la premisa, no la táctica) |
| 5 | Lista de virales para revisión cualitativa, por antigüedad | Sí (como listado) |
| 6 | Fichajes en ventana de mercado | Sí, con supuesto de meses de ventana |
| 7 | Rellenar huecos vs. descartar categorías débiles | No comprobable empíricamente |
| 8 | Top por retención (Shorts/Largos) | Sí |
| 9 | Mapa de correlaciones | Sí |

**Requisito previo:** base de datos `golazo_growup` cargada.

## 0. Carga de datos y columnas derivadas

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

from src import db_connection as dbc

url = dbc.DATABASE_URL or (
    f"postgresql+psycopg2://{dbc.DB_CONFIG['user']}:{dbc.DB_CONFIG['password']}"
    f"@{dbc.DB_CONFIG['host']}:{dbc.DB_CONFIG['port']}/{dbc.DB_CONFIG['dbname']}"
)
if url.startswith('postgresql://'):
    url = url.replace('postgresql://', 'postgresql+psycopg2://', 1)
engine = create_engine(url)

video = pd.read_sql('SELECT * FROM video', engine, parse_dates=['fecha_publicacion'])
retencion = pd.read_sql('SELECT * FROM retencion_audiencia', engine)
evolucion = pd.read_sql('SELECT * FROM evolucion_diaria', engine, parse_dates=['fecha'])

ORDEN_DIAS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
HOY = pd.Timestamp.now().normalize()

evolucion['dia_semana'] = evolucion['fecha'].dt.day_name()
evolucion['suscriptores_netos'] = evolucion['subscribers_gained'] - evolucion['subscribers_lost']

UMBRAL_SHORT_SEGUNDOS = 180
video['es_short'] = video['duracion_segundos'] <= UMBRAL_SHORT_SEGUNDOS
video['formato'] = video['es_short'].map({True: 'Short', False: 'Largo'})
video['dia_semana_publicacion'] = video['fecha_publicacion'].dt.day_name()
video['mes_publicacion'] = video['fecha_publicacion'].dt.month

video['antiguedad_dias'] = (HOY - video['fecha_publicacion']).dt.days.clip(lower=1)
video['views_por_dia'] = video['views_totales'] / video['antiguedad_dias']
MIN_DIAS_PARA_VELOCIDAD = 7
video['apto_velocidad'] = video['antiguedad_dias'] >= MIN_DIAS_PARA_VELOCIDAD

video['ratio_likes_vista'] = video['likes'] / video['views_totales']
video['ratio_comentarios_vista'] = video['comentarios'] / video['views_totales']
video['z_views_categoria_formato'] = video.groupby(['categoria', 'formato'])['views_totales'].transform(
    lambda s: (s - s.mean()) / s.std() if s.std() > 0 else 0
)

retencion_media_por_video = retencion.groupby('video_id')['audience_watch_ratio'].mean().rename('retencion_media')
video = video.merge(retencion_media_por_video, on='video_id', how='left')

patron_semanal_audiencia = evolucion.groupby('dia_semana')['views'].mean().reindex(ORDEN_DIAS)
dias_fuertes = patron_semanal_audiencia.sort_values(ascending=False).head(2).index.tolist()
video['es_dia_fuerte'] = video['dia_semana_publicacion'].isin(dias_fuertes)

# Proxy de 'día de competición europea entre semana' — supuesto de dominio, no un dato real
DIAS_EUROPA = ['Tuesday', 'Wednesday']
video['es_dia_europeo'] = video['dia_semana_publicacion'].isin(DIAS_EUROPA)
video['dia_relevante_partido'] = video['es_dia_fuerte'] | video['es_dia_europeo']

print(f'Vídeos: {len(video)} | Días fuertes: {dias_fuertes} | Días "europeos" asumidos: {DIAS_EUROPA}')

## 1. Shorts de 'Opinión post-partido': ¿se concentran en días de partido?

**Limitación explícita:** no tenemos hora de publicación ni calendario real de partidos. Se aproxima 'día de partido' como fin de semana (findes fuertes reales del canal) + martes/miércoles (proxy estándar de competición europea). Es una aproximación de dominio, no una medición directa de cercanía al pitido final.

In [ ]:
opinion_shorts = video[(video['categoria'] == 'Opinión post-partido') & (video['formato'] == 'Short')]
print(f'Shorts de Opinión post-partido: {len(opinion_shorts)}')

comparativa_opinion_dias = opinion_shorts.groupby('dia_relevante_partido')[
    ['views_totales', 'ratio_likes_vista', 'ratio_comentarios_vista', 'retencion_media']
].mean()
comparativa_opinion_dias.index = comparativa_opinion_dias.index.map({True: 'Día de partido (proxy)', False: 'Resto de días'})
print('Opinión (Short): rendimiento en días de partido (proxy) vs. resto:')
display(comparativa_opinion_dias.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
comparativa_opinion_dias['views_totales'].plot(kind='bar', color=['#B23A2E', '#2E6E9E'], ax=ax)
ax.set_ylabel('Vistas medias')
ax.set_title("Opinión post-partido (Short): días de partido (proxy) vs. resto")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Comentarios: en el propio feed, la inmediatez del evento se traduce en más comentarios por vista
# (discusión activa recién acabado el partido) — es la métrica más alineada con tu hipótesis de 'apoyo/confrontación de opinión'
mejora_comentarios = (
    comparativa_opinion_dias.loc['Día de partido (proxy)', 'ratio_comentarios_vista']
    / comparativa_opinion_dias.loc['Resto de días', 'ratio_comentarios_vista'] - 1
) * 100
print(f"Comentarios/vista en días de partido (proxy) vs. resto: {mejora_comentarios:+.1f}%")
if mejora_comentarios > 10:
    print('Consistente con la hipótesis: más conversación activa en días de partido — refuerza publicar '
          'Shorts de opinión pegados a esos días.')
else:
    print('No hay una diferencia clara de comentarios entre días de partido y el resto con el proxy actual '
          '— para confirmar esta hipótesis con más precisión haría falta la hora real de publicación.')

## 2. Largo de 'Seguimiento del club': ¿es la categoría que mejor funciona, siempre?

Se comprueba el ranking de TODAS las categorías (formato Largo) tanto en día fuerte como en el resto de la semana, para confirmar o refutar 'casi siempre tendríamos un vídeo largo sobre la categoría que mejor funciona tanto en día fuerte como en el resto'.

In [ ]:
largos = video[video['formato'] == 'Largo']
ranking_dia_fuerte = largos[largos['es_dia_fuerte']].groupby('categoria')['views_totales'].mean().sort_values(ascending=False)
ranking_resto = largos[~largos['es_dia_fuerte']].groupby('categoria')['views_totales'].mean().sort_values(ascending=False)

print('Ranking de categorías (Largo) en DÍA FUERTE:')
display(ranking_dia_fuerte.round(0))
print('\nRanking de categorías (Largo) en RESTO de la semana:')
display(ranking_resto.round(0))

top_fuerte = ranking_dia_fuerte.idxmax()
top_resto = ranking_resto.idxmax()
print(f"\n¿Seguimiento del club es líder en ambos? "
      f"Día fuerte: {'SÍ' if top_fuerte == 'Seguimiento del club' else f'NO (lidera {top_fuerte})'} | "
      f"Resto: {'SÍ' if top_resto == 'Seguimiento del club' else f'NO (lidera {top_resto})'}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
pd.DataFrame({'Día fuerte': ranking_dia_fuerte, 'Resto de la semana': ranking_resto}).plot(
    kind='barh', ax=ax, color=['#F2A65A', '#2E6E9E']
)
ax.set_xlabel('Vistas medias')
ax.set_title('Ranking de categorías (Largo): día fuerte vs. resto de la semana')
plt.tight_layout()
plt.show()

## 3. Curiosidades de fútbol a fondo — y comparación directa con Afición

In [ ]:
curiosidades = video[video['categoria'] == 'Curiosidades de fútbol']
aficion = video[video['categoria'] == 'Afición']
print(f'Curiosidades de fútbol: {len(curiosidades)} vídeos | Afición: {len(aficion)} vídeos '
      f'(diferencia: {len(curiosidades) - len(aficion)})')

metricas = ['views_totales', 'likes', 'comentarios', 'ratio_likes_vista', 'ratio_comentarios_vista', 'retencion_media']
comparativa_curiosidades = pd.DataFrame({
    'Curiosidades': curiosidades[metricas].mean(),
    'Afición': aficion[metricas].mean(),
    'Media del canal': video[metricas].mean(),
})
display(comparativa_curiosidades.round(3))

In [ ]:
# ¿Es un tipo de contenido más reciente en la producción del canal?
antiguedad_media_categoria = video.groupby('categoria')['antiguedad_dias'].mean().sort_values()
print('Antigüedad media por categoría (menor = más reciente en la producción del canal):')
display(antiguedad_media_categoria.round(0))

rank_curiosidades = list(antiguedad_media_categoria.index).index('Curiosidades de fútbol') + 1
print(f"\nCuriosidades de fútbol ocupa la posición {rank_curiosidades} de {len(antiguedad_media_categoria)} "
      f"por antigüedad media ({'entre las más recientes' if rank_curiosidades <= 2 else 'sin patrón claro de recencia'}).")

In [ ]:
# Ranking de categorías por vistas medias (confirma o refuta 'más visualizaciones de media')
ranking_categorias_views = video.groupby('categoria')['views_totales'].mean().sort_values(ascending=False)
print('Ranking de categorías por vistas medias:')
display(ranking_categorias_views.round(0))

fig, ax = plt.subplots(figsize=(8, 4))
colores = ['#F2A65A' if c == 'Curiosidades de fútbol' else ('#B23A2E' if c == 'Afición' else '#2E6E9E')
           for c in ranking_categorias_views.index]
ranking_categorias_views.plot(kind='barh', color=colores, ax=ax)
ax.set_xlabel('Vistas medias')
ax.set_title('Ranking de categorías por vistas medias (naranja=Curiosidades, rojo=Afición)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Ganancia neta de suscriptores para Curiosidades — solo posible si la fecha de publicación
# cae dentro de la ventana de 30 días de evolucion_diaria (limitación ya conocida)
fecha_min, fecha_max = evolucion['fecha'].min(), evolucion['fecha'].max()
curiosidades_en_ventana = curiosidades[
    (curiosidades['fecha_publicacion'] >= fecha_min) & (curiosidades['fecha_publicacion'] <= fecha_max)
]
print(f"Vídeos de Curiosidades dentro de la ventana de 30 días (se puede cruzar con suscriptores): "
      f"{len(curiosidades_en_ventana)} de {len(curiosidades)}")

if len(curiosidades_en_ventana):
    ganancias = []
    for _, v in curiosidades_en_ventana.iterrows():
        ventana_dias = evolucion[
            (evolucion['fecha'] >= v['fecha_publicacion']) & (evolucion['fecha'] <= v['fecha_publicacion'] + pd.Timedelta(days=3))
        ]
        ganancias.append(ventana_dias['suscriptores_netos'].sum())
    print(f"Ganancia neta media de suscriptores en los 3 días siguientes a un vídeo de Curiosidades: "
          f"{np.mean(ganancias):.1f}")
else:
    print('Ningún vídeo de Curiosidades cae en la ventana de 30 días disponible — no se puede calcular '
          'esta cifra con los datos actuales.')

## 4. Shorts de 'Afición' como relleno de bajo coste — ¿se sostiene la premisa?

Solo se puede confirmar la premisa (Afición rinde peor, así que el formato de menor coste de oportunidad es el adecuado); la táctica en sí (mezclar curiosidades de afición) no tiene histórico que comprobar.

In [ ]:
resto_categorias = video[video['categoria'] != 'Afición']
print(f"Vistas medias — Afición: {aficion['views_totales'].mean():.0f} | "
      f"Resto del canal: {resto_categorias['views_totales'].mean():.0f}")
print(f"Afición es la categoría con {'MENOS' if ranking_categorias_views.idxmin() == 'Afición' else 'NO la menos'} "
      f"vistas medias del canal.")

coste_oportunidad_short = aficion[aficion['formato'] == 'Short']['duracion_segundos'].mean()
coste_oportunidad_largo = aficion[aficion['formato'] == 'Largo']['duracion_segundos'].mean() if len(aficion[aficion['formato']=='Largo']) else float('nan')
print(f"\nDuración media si se hiciera en Short: {coste_oportunidad_short/60:.1f} min "
      f"vs. Largo: {coste_oportunidad_largo/60:.1f} min — el Short consume menos ventana de producción "
      f"y de audiencia por la misma categoría de bajo rendimiento.")

## 5. Vídeos virales para revisión cualitativa del cliente (ordenados por antigüedad)

Listado accionable: los vídeos que MENOS tiempo llevan publicados dentro de los que ya han demostrado ser virales — son los que mejor reflejan 'qué está funcionando ahora', dentro de la limitación de 30 días de la ventana de audiencia disponible.

In [ ]:
picos_virales = video[video['z_views_categoria_formato'] > 2]
virales_para_revision = picos_virales.sort_values('antiguedad_dias').copy()
print(f'Vídeos virales detectados: {len(virales_para_revision)} — ordenados de más reciente a más antiguo:')
display(virales_para_revision[
    ['titulo', 'categoria', 'formato', 'fecha_publicacion', 'dia_semana_publicacion',
     'antiguedad_dias', 'views_totales', 'retencion_media']
].round(2))

## 6. Categoría 'Fichajes': ¿rinde mejor en ventana de mercado?

**Supuesto de dominio** (no hay campo real de 'ventana de fichajes' en los datos): se asume ventana de mercado en enero (invierno) y junio-agosto (verano), meses típicos en el fútbol europeo.

In [ ]:
MESES_VENTANA_FICHAJES = [1, 6, 7, 8]
fichajes = video[video['categoria'] == 'Fichajes'].copy()
fichajes['en_ventana'] = fichajes['mes_publicacion'].isin(MESES_VENTANA_FICHAJES)

print(f'Vídeos de Fichajes: {len(fichajes)} | En ventana: {fichajes["en_ventana"].sum()} | '
      f'Fuera de ventana: {(~fichajes["en_ventana"]).sum()}')

comparativa_fichajes = fichajes.groupby('en_ventana')[
    ['views_totales', 'ratio_likes_vista', 'ratio_comentarios_vista', 'retencion_media']
].mean()
comparativa_fichajes.index = comparativa_fichajes.index.map({True: 'En ventana', False: 'Fuera de ventana'})
display(comparativa_fichajes.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
comparativa_fichajes['views_totales'].plot(kind='bar', color=['#2E6E9E', '#B23A2E'], ax=ax)
ax.set_ylabel('Vistas medias')
ax.set_title('Fichajes: en ventana de mercado vs. fuera')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Categorías débiles: ¿rellenar huecos o descartar?

No es una hipótesis comprobable empíricamente con datos históricos — es una decisión de capacidad de producción. Lo único que los datos pueden aportar es un ranking objetivo de qué categorías son consistentemente más débiles, para apoyar la decisión.

In [ ]:
ranking_completo = video.groupby('categoria').agg(
    n_videos=('video_id', 'count'),
    views_medias=('views_totales', 'mean'),
    retencion_media=('retencion_media', 'mean'),
).sort_values('views_medias')
print('Ranking de categorías, de más débil a más fuerte (candidatas a relleno o descarte, de arriba a abajo):')
display(ranking_completo.round(3))

## 8. Top por retención (Shorts y Largos) — la palanca económica real

In [ ]:
for formato in ['Short', 'Largo']:
    top_retencion = video[video['formato'] == formato].sort_values('retencion_media', ascending=False).head(10)
    print(f'\nTOP 10 {formato}s por RETENCIÓN:')
    display(top_retencion[
        ['titulo', 'categoria', 'duracion_segundos', 'views_totales', 'retencion_media']
    ].round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, formato, color in zip(axes, ['Short', 'Largo'], ['#B23A2E', '#2E6E9E']):
    top10 = video[video['formato'] == formato].sort_values('retencion_media', ascending=False).head(10)
    ax.barh(top10['titulo'].str[:30], top10['retencion_media'] * 100, color=color)
    ax.set_xlabel('Retención media (%)')
    ax.set_title(f'Top 10 {formato}s por retención')
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Mapa de correlaciones

Variables incluidas: vistas, likes, comentarios, retención, duración, antigüedad, día de la semana (numérico), formato y día fuerte (como indicadores). Se añaden `antiguedad_dias` y `es_dia_fuerte` porque son las dos variables que más han condicionado las lecturas de este notebook y el anterior.

In [ ]:
df_corr = video.copy()
df_corr['dia_semana_num'] = df_corr['dia_semana_publicacion'].map({d: i for i, d in enumerate(ORDEN_DIAS)})
df_corr['es_short_num'] = df_corr['es_short'].astype(int)
df_corr['es_dia_fuerte_num'] = df_corr['es_dia_fuerte'].astype(int)

columnas_correlacion = [
    'views_totales', 'likes', 'comentarios', 'retencion_media', 'duracion_segundos',
    'antiguedad_dias', 'dia_semana_num', 'es_short_num', 'es_dia_fuerte_num',
    'ratio_likes_vista', 'ratio_comentarios_vista',
]
matriz_corr = df_corr[columnas_correlacion].corr().round(2)
display(matriz_corr)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(matriz_corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(columnas_correlacion)))
ax.set_yticks(range(len(columnas_correlacion)))
ax.set_xticklabels(columnas_correlacion, rotation=45, ha='right')
ax.set_yticklabels(columnas_correlacion)
for i in range(len(columnas_correlacion)):
    for j in range(len(columnas_correlacion)):
        ax.text(j, i, f'{matriz_corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, label='Correlación')
plt.title('Mapa de correlaciones — variables de vídeo')
plt.tight_layout()
plt.show()

## Conclusión: ¿hace falta regenerar los datos sintéticos?

*(Completar tras revisar los resultados reales de tu ejecución.)*

Con los datos actuales SÍ se pueden defender en una presentación: el ranking de categorías (punto 2 y 7), el análisis de Curiosidades vs. Afición (punto 3), Fichajes en ventana (punto 6, con el supuesto explícito), el Top por retención (punto 8) y el mapa de correlaciones (punto 9).

Lo que se queda corto y **sí justificaría regenerar/ampliar los datos sintéticos** si se quiere defender con más fuerza en la demo:
- **Hora exacta de publicación** (cambiar `fecha_publicacion` de `DATE` a `TIMESTAMP` en el esquema) para probar de verdad la hipótesis 1 (cercanía al pitido final).
- **Un calendario de partidos simulado** (fecha, si fue victoria/derrota/empate, si fue rival directo o competición europea) para poder generar retención/vistas correlacionadas con el resultado real del partido, no solo con el día de la semana — esto haría mucho más rica y defendible la hipótesis 1 y reforzaría la 2.
- **Ingresos por vídeo** (dimensión `video` en la Analytics API, ya comentado en fases anteriores) si se quiere hablar de rentabilidad por categoría, no solo de vistas/retención.